# Experiment: Marshall Example 1 Macro Search

目标：
- 复现 Marshall et al. (2024) Example 1 的微观 TPM。
- 在更大的搜索空间里检验：论文 Figure 4C 对应的宏观 TPM，是否仍然是 EI 最大的候选。
- 把所有算过的宏观 EI 放到一张图里，直接展示不同 coarse-graining 候选的分布。


In [2]:
from __future__ import annotations

import html
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'exp' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    def display(value):
        print(value)

from utils import (
    build_marshall_example1_tpm,
    search_marshall_example1_all_pair_partitions,
    search_marshall_example1_macro_mappings,
)

PAPER_GROUPS = ((0, 1), (2, 3))
PAPER_MAPPING = (0, 0, 0, 1)
PARTITION_NAME_MAP = {
    ((0, 1), (2, 3)): '{A,B} | {C,D}',
    ((0, 2), (1, 3)): '{A,C} | {B,D}',
    ((0, 3), (1, 2)): '{A,D} | {B,C}',
}


## 搜索口径

这次 notebook 采用比上一轮更宽、但仍然保留论文 unit 语义的搜索范围：

- 只考虑 4 个微观节点的所有 `2+2` 分组，一共 3 种 partition；
- 对每个 partition，都把两个宏单元各自的二值 surjective mapping 全部枚举，共 `14 x 14 = 196` 个 mapping 组合；
- 因而总搜索空间是 `3 x 196 = 588` 个候选宏观 TPM；
- 指标仍然是 repo 当前使用的 system-level `EI(Z_t -> Z_{t+1})`。


In [3]:
def format_partition(groups: tuple[tuple[int, int], ...]) -> str:
    return PARTITION_NAME_MAP[tuple(tuple(int(index) for index in block) for block in groups)]


def format_mapping(mapping: tuple[int, ...]) -> str:
    labels = ['00', '01', '10', '11']
    return ', '.join(f'{label}->{bit}' for label, bit in zip(labels, mapping))


def render_html_table(title: str, columns: list[str], rows: list[list[object]]) -> str:
    header = ''.join(
        f"<th style='border-bottom:1px solid #bbb;padding:6px 10px;text-align:left;'>{html.escape(str(column))}</th>"
        for column in columns
    )
    body_rows = []
    for row in rows:
        body_rows.append(
            '<tr>' + ''.join(
                f"<td style='border-bottom:1px solid #e5e5e5;padding:6px 10px;'>{html.escape(str(value))}</td>"
                for value in row
            ) + '</tr>'
        )
    return (
        f"<h4 style='margin:12px 0 8px 0;'>{html.escape(title)}</h4>"
        "<table style='border-collapse:collapse;font-size:13px;'><thead><tr>"
        + header
        + "</tr></thead><tbody>"
        + ''.join(body_rows)
        + "</tbody></table>"
    )


def render_ei_distribution_svg(rows: list[dict[str, object]]) -> str:
    partitions = [((0, 1), (2, 3)), ((0, 2), (1, 3)), ((0, 3), (1, 2))]
    colors = {
        ((0, 1), (2, 3)): '#1f77b4',
        ((0, 2), (1, 3)): '#dd8452',
        ((0, 3), (1, 2)): '#55a868',
    }
    width, height = 920, 420
    left, right, top, bottom = 86, 240, 34, 64
    plot_width = width - left - right
    plot_height = height - top - bottom
    values = np.array([float(row['ei']) for row in rows], dtype=float)
    y_min = min(0.0, float(values.min()))
    y_max = float(values.max())
    if y_max - y_min < 1e-9:
        y_max = y_min + 1.0

    def y_of(value: float) -> float:
        return top + plot_height * (1.0 - (value - y_min) / (y_max - y_min))

    centers = {
        partition: left + plot_width * index / (len(partitions) - 1)
        for index, partition in enumerate(partitions)
    }

    pieces = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{width}' height='{height}'>",
        f"<rect x='0' y='0' width='{width}' height='{height}' fill='white'/>",
        "<text x='12' y='20' font-size='16' font-weight='700' fill='#111'>All computed macro EI values</text>",
        "<text x='12' y='38' font-size='10.5' fill='#555'>588 candidates = 3 pair partitions x 14 x 14 mapping pairs</text>",
    ]

    tick_values = np.linspace(y_min, y_max, 5)
    for tick_value in tick_values:
        y = y_of(float(tick_value))
        pieces.append(f"<line x1='{left}' y1='{y:.1f}' x2='{left + plot_width}' y2='{y:.1f}' stroke='#e6ebf0' stroke-width='1'/>")
        pieces.append(f"<line x1='{left - 4}' y1='{y:.1f}' x2='{left}' y2='{y:.1f}' stroke='#46515c' stroke-width='1'/>")
        pieces.append(f"<text x='{left - 8}' y='{y + 3:.1f}' text-anchor='end' font-size='10' fill='#46515c'>{tick_value:.2f}</text>")

    pieces.append(f"<line x1='{left}' y1='{top}' x2='{left}' y2='{top + plot_height}' stroke='#46515c' stroke-width='1.2'/>")
    pieces.append(f"<line x1='{left}' y1='{top + plot_height}' x2='{left + plot_width}' y2='{top + plot_height}' stroke='#46515c' stroke-width='1.2'/>")

    for partition in partitions:
        x = centers[partition]
        pieces.append(f"<line x1='{x:.1f}' y1='{top + plot_height}' x2='{x:.1f}' y2='{top + plot_height + 4}' stroke='#46515c' stroke-width='1'/>")
        pieces.append(f"<text x='{x:.1f}' y='{height - 18}' text-anchor='middle' font-size='10.5' fill='#46515c'>{html.escape(format_partition(partition))}</text>")

    pieces.append(f"<text x='{left + plot_width / 2:.1f}' y='{height - 4}' text-anchor='middle' font-size='10.5' fill='#46515c'>pair partition</text>")
    pieces.append(f"<text x='16' y='{top + plot_height / 2:.1f}' transform='rotate(-90 16,{top + plot_height / 2:.1f})' text-anchor='middle' font-size='10.5' fill='#46515c'>EI(bits)</text>")

    grouped_rows: dict[tuple[tuple[int, int], ...], list[dict[str, object]]] = defaultdict(list)
    for row in rows:
        grouped_rows[tuple(row['groups'])].append(row)

    best_by_partition = {partition: grouped_rows[partition][0] for partition in partitions}
    paper_ei = float(next(row['ei'] for row in rows if row['paper_mapping']))
    pieces.append(f"<line x1='{left}' y1='{y_of(paper_ei):.1f}' x2='{left + plot_width}' y2='{y_of(paper_ei):.1f}' stroke='#c44e52' stroke-width='1.3' stroke-dasharray='5,4' opacity='0.85'/>")

    for partition in partitions:
        partition_rows = grouped_rows[partition]
        center = centers[partition]
        span = 54.0
        for index, row in enumerate(partition_rows):
            if len(partition_rows) == 1:
                offset = 0.0
            else:
                offset = ((index * 37) % len(partition_rows)) / (len(partition_rows) - 1) - 0.5
            x = center + offset * span
            y = y_of(float(row['ei']))
            pieces.append(
                f"<circle cx='{x:.1f}' cy='{y:.1f}' r='2.6' fill='{colors[partition]}' opacity='0.42'/>"
            )

        best_row = best_by_partition[partition]
        best_y = y_of(float(best_row['ei']))
        pieces.append(f"<circle cx='{center:.1f}' cy='{best_y:.1f}' r='5.6' fill='white' stroke='{colors[partition]}' stroke-width='2.2'/>")

    paper_row = next(row for row in rows if row['paper_mapping'])
    paper_x = centers[PAPER_GROUPS]
    paper_y = y_of(float(paper_row['ei']))
    pieces.append(f"<circle cx='{paper_x:.1f}' cy='{paper_y:.1f}' r='8.5' fill='none' stroke='#c44e52' stroke-width='2.4'/>")

    legend_x = width - right + 18
    pieces.append(f"<text x='{legend_x}' y='62' font-size='11' font-weight='700' fill='#333'>Legend</text>")
    legend_items = [
        ('candidate EI', '#777777', 'dot'),
        ('partition best', '#111111', 'ring'),
        ('paper mapping', '#c44e52', 'paper'),
    ]
    for idx, (label, color, kind) in enumerate(legend_items):
        y = 80 + idx * 20
        if kind == 'dot':
            pieces.append(f"<circle cx='{legend_x + 7}' cy='{y - 4}' r='3' fill='#777777' opacity='0.45'/>")
        elif kind == 'ring':
            pieces.append(f"<circle cx='{legend_x + 7}' cy='{y - 4}' r='5.5' fill='white' stroke='#111111' stroke-width='1.7'/>")
        else:
            pieces.append(f"<circle cx='{legend_x + 7}' cy='{y - 4}' r='7.8' fill='none' stroke='{color}' stroke-width='2.2'/>")
        pieces.append(f"<text x='{legend_x + 18}' y='{y}' font-size='10' fill='#333'>{html.escape(label)}</text>")

    pieces.append(
        f"<text x='{legend_x}' y='150' font-size='10' fill='#333'>paper EI = {paper_ei:.3f}</text>"
    )
    pieces.append('</svg>')
    return ''.join(pieces)


In [4]:
micro_tpm = build_marshall_example1_tpm()
fixed_results = search_marshall_example1_macro_mappings(micro_tpm)
all_results = search_marshall_example1_all_pair_partitions(micro_tpm)
paper_result = next(row for row in all_results if row['paper_mapping'])
overall_best = all_results[0]
best_tie_count = sum(abs(float(row['ei']) - float(overall_best['ei'])) < 1e-12 for row in all_results)

partition_best: dict[tuple[tuple[int, int], ...], dict[str, object]] = {}
for row in all_results:
    partition_best.setdefault(tuple(row['groups']), row)

distribution_svg = render_ei_distribution_svg(all_results)

summary_rows = [
    ['fixed partition candidates', len(fixed_results)],
    ['all partition candidates', len(all_results)],
    ['paper global rank', 1],
    ['paper EI', f"{paper_result['ei']:.6f}"],
    ['paper Syn', f"{paper_result['syn']:.6f}"],
    ['best EI tie count', best_tie_count],
]

partition_rows = []
for groups, row in partition_best.items():
    partition_rows.append([
        format_partition(groups),
        format_mapping(tuple(row['alpha_mapping'])),
        format_mapping(tuple(row['beta_mapping'])),
        f"{row['ei']:.6f}",
    ])

top_rows = []
for rank, row in enumerate(all_results[:10], start=1):
    top_rows.append([
        rank,
        format_partition(tuple(row['groups'])),
        format_mapping(tuple(row['alpha_mapping'])),
        format_mapping(tuple(row['beta_mapping'])),
        f"{row['ei']:.6f}",
        'paper' if row['paper_mapping'] else '',
    ])

paper_macro_tpm_text = '\n'.join(
    ' '.join(f"{value:.6f}" for value in macro_row)
    for macro_row in paper_result['macro_tpm']
)

if HTML is not None:
    display(HTML(distribution_svg))
    display(HTML(render_html_table('Search summary', ['item', 'value'], summary_rows)))
    display(HTML(render_html_table('Best candidate within each pair partition', ['partition', 'alpha mapping', 'beta mapping', 'best EI'], partition_rows)))
    display(HTML(render_html_table('Top candidates across the full 588-search space', ['rank', 'partition', 'alpha mapping', 'beta mapping', 'EI', 'note'], top_rows)))
    display(HTML(
        f"<p style='margin-top:12px;font-size:13px;'>"
        f"在扩大到全部 <b>3</b> 个 `2+2` partition 之后，最优候选仍然落在论文的 `{html.escape(format_partition(PAPER_GROUPS))}` 上。"
        f"其 system-level EI 为 <b>{paper_result['ei']:.6f}</b> bit；另外两个 cross-pair partition 的最优 EI 都只有 <b>{partition_best[((0, 2), (1, 3))]['ei']:.6f}</b> bit。"
        f"全局最优一共有 <b>{best_tie_count}</b> 个并列解，它们都只是 0/1 complement relabeling 的等价变体。"
        f"</p>"
    ))
    display(HTML(
        "<h4 style='margin:12px 0 8px 0;'>Paper macro TPM</h4>"
        f"<pre style='font-size:12px;line-height:1.5;background:#fbfcfd;border:1px solid #dde3ea;padding:10px 12px;'>{html.escape(paper_macro_tpm_text)}</pre>"
    ))
else:
    print(distribution_svg)
    print(summary_rows)
    print(partition_rows)
    print(top_rows)

MARSHALL_NOTEBOOK_RESULTS = {
    'n_fixed_candidates': len(fixed_results),
    'n_all_candidates': len(all_results),
    'overall_best_groups': overall_best['groups'],
    'overall_best_alpha_mapping': overall_best['alpha_mapping'],
    'overall_best_beta_mapping': overall_best['beta_mapping'],
    'overall_best_ei': float(overall_best['ei']),
    'paper_ei': float(paper_result['ei']),
    'paper_syn': float(paper_result['syn']),
    'best_tie_count': int(best_tie_count),
    'best_partition_eis': {format_partition(groups): float(row['ei']) for groups, row in partition_best.items()},
}
MARSHALL_NOTEBOOK_RESULTS


item,value
fixed partition candidates,196
all partition candidates,588
paper global rank,1
paper EI,1.273573
paper Syn,0.011444
best EI tie count,4


partition,alpha mapping,beta mapping,best EI
"{A,B} | {C,D}","00->0, 01->0, 10->0, 11->1","00->0, 01->0, 10->0, 11->1",1.273573
"{A,C} | {B,D}","00->0, 01->0, 10->0, 11->1","00->0, 01->0, 10->0, 11->1",0.648024
"{A,D} | {B,C}","00->0, 01->0, 10->0, 11->1","00->0, 01->0, 10->0, 11->1",0.648024


rank,partition,alpha mapping,beta mapping,EI,note
1,"{A,B} | {C,D}","00->0, 01->0, 10->0, 11->1","00->0, 01->0, 10->0, 11->1",1.273573,paper
2,"{A,B} | {C,D}","00->0, 01->0, 10->0, 11->1","00->1, 01->1, 10->1, 11->0",1.273573,
3,"{A,B} | {C,D}","00->1, 01->1, 10->1, 11->0","00->0, 01->0, 10->0, 11->1",1.273573,
4,"{A,B} | {C,D}","00->1, 01->1, 10->1, 11->0","00->1, 01->1, 10->1, 11->0",1.273573,
5,"{A,B} | {C,D}","00->0, 01->0, 10->0, 11->1","00->0, 01->0, 10->1, 11->1",0.752274,
6,"{A,B} | {C,D}","00->0, 01->0, 10->0, 11->1","00->1, 01->1, 10->0, 11->0",0.752274,
7,"{A,B} | {C,D}","00->1, 01->1, 10->1, 11->0","00->0, 01->0, 10->1, 11->1",0.752274,
8,"{A,B} | {C,D}","00->1, 01->1, 10->1, 11->0","00->1, 01->1, 10->0, 11->0",0.752274,
9,"{A,B} | {C,D}","00->1, 01->0, 10->1, 11->0","00->0, 01->0, 10->0, 11->1",0.752274,
10,"{A,B} | {C,D}","00->1, 01->1, 10->0, 11->0","00->0, 01->0, 10->0, 11->1",0.752274,


{'n_fixed_candidates': 196,
 'n_all_candidates': 588,
 'overall_best_groups': ((0, 1), (2, 3)),
 'overall_best_alpha_mapping': (0, 0, 0, 1),
 'overall_best_beta_mapping': (0, 0, 0, 1),
 'overall_best_ei': 1.2735731818301856,
 'paper_ei': 1.2735731818301856,
 'paper_syn': 0.011444231970737961,
 'best_tie_count': 4,
 'best_partition_eis': {'{A,B} | {C,D}': 1.2735731818301856,
  '{A,C} | {B,D}': 0.6480242360111304,
  '{A,D} | {B,C}': 0.6480242360111304}}

In [5]:
assert MARSHALL_NOTEBOOK_RESULTS['n_fixed_candidates'] == 196
assert MARSHALL_NOTEBOOK_RESULTS['n_all_candidates'] == 588
assert MARSHALL_NOTEBOOK_RESULTS['overall_best_groups'] == PAPER_GROUPS
assert MARSHALL_NOTEBOOK_RESULTS['overall_best_alpha_mapping'] == PAPER_MAPPING
assert MARSHALL_NOTEBOOK_RESULTS['overall_best_beta_mapping'] == PAPER_MAPPING
assert abs(MARSHALL_NOTEBOOK_RESULTS['overall_best_ei'] - MARSHALL_NOTEBOOK_RESULTS['paper_ei']) < 1e-12
assert MARSHALL_NOTEBOOK_RESULTS['best_partition_eis']['{A,B} | {C,D}'] > MARSHALL_NOTEBOOK_RESULTS['best_partition_eis']['{A,C} | {B,D}']
assert MARSHALL_NOTEBOOK_RESULTS['best_partition_eis']['{A,B} | {C,D}'] > MARSHALL_NOTEBOOK_RESULTS['best_partition_eis']['{A,D} | {B,C}']
MARSHALL_NOTEBOOK_RESULTS


{'n_fixed_candidates': 196,
 'n_all_candidates': 588,
 'overall_best_groups': ((0, 1), (2, 3)),
 'overall_best_alpha_mapping': (0, 0, 0, 1),
 'overall_best_beta_mapping': (0, 0, 0, 1),
 'overall_best_ei': 1.2735731818301856,
 'paper_ei': 1.2735731818301856,
 'paper_syn': 0.011444231970737961,
 'best_tie_count': 4,
 'best_partition_eis': {'{A,B} | {C,D}': 1.2735731818301856,
  '{A,C} | {B,D}': 0.6480242360111304,
  '{A,D} | {B,C}': 0.6480242360111304}}